# 🧪 Phase 10: Evaluation, Validation, and Testing

## Objectives
This notebook performs comprehensive evaluation and validation of the trained H5-OmniFusion model:

| Test Category | Description |
|---------------|-------------|
| **Data Integrity** | Validate H5 files, labels, and 108-step compliance |
| **Model Validation** | Load checkpoints and verify model integrity |
| **Performance Metrics** | F1, Precision, Recall, Accuracy, AUC-ROC |
| **Threshold Analysis** | Optimize decision boundary for clinical use |
| **Visualization** | Confusion matrix, ROC curves |

## Configuration
- **Checkpoints**: Phase 10 trained models
- **Tier**: Medium (11.98M params)
- **Data Sources**: Primary, Extended, and Supplementary corpora

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Clone/Update Repository

In [ ]:
import os

if not os.path.exists('/content/phase2'):
    !git clone https://github.com/nithin12342/phase2.git /content/phase2
    print("✅ Repository cloned successfully!")
else:
    %cd /content/phase2
    !git fetch origin
    !git reset --hard origin/main
    %cd /content
    print("✅ Repository updated to latest!")

## Step 3: Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio --quiet
!pip install transformers h5py pandas scikit-learn tqdm matplotlib seaborn --quiet
print("✅ Dependencies installed!")

## Step 4: Configuration

In [ ]:
# ============================================
# Phase 10 Evaluation Configuration
# ============================================

import os
import sys
from pathlib import Path

# Core paths
OS_PATH = "/content/phase2/ml_pipeline/h5_omnifusion"
DATA_ROOT = "/content/drive/MyDrive/DAIC-WOZ_Datasets"
DATA_DIR = f"{DATA_ROOT}/H5_OmniFusion_Output"
LABELS = f"{DATA_DIR}/all_labels.csv"
CHECKPOINT_DIR = f"{DATA_ROOT}/checkpoints_phase10"

# Model configuration
TIER = "medium"
BATCH_SIZE = 16
MAX_SEQ_LEN = 256

# Add project to path
sys.path.insert(0, OS_PATH)

# Validate paths
print("📁 Path Verification:")
print(f"   OS_PATH exists: {os.path.exists(OS_PATH)}")
print(f"   DATA_DIR exists: {os.path.exists(DATA_DIR)}")
print(f"   LABELS exists: {os.path.exists(LABELS)}")
print(f"   CHECKPOINT_DIR exists: {os.path.exists(CHECKPOINT_DIR)}")

print(f"\n⚙️ Configuration:")
print(f"   Tier: {TIER}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Max Sequence Length: {MAX_SEQ_LEN}")

## Step 5: Dataset Validation & Integrity Checks

In [ ]:
import pandas as pd
import numpy as np
import glob

print("🔍 Dataset Validation...\n")

# Load labels
if os.path.exists(LABELS):
    df = pd.read_csv(LABELS)
    print(f"📊 Total Samples in Labels: {len(df)}")
    
    # Check required columns
    required_cols = ['Participant_ID']
    optional_cols = ['PHQ8_Score', 'PHQ8_Binary', 'Depression_Label', 'Source']
    
    missing_required = [c for c in required_cols if c not in df.columns]
    if missing_required:
        print(f"❌ Missing required columns: {missing_required}")
    else:
        print("✅ Required columns present")
    
    present_optional = [c for c in optional_cols if c in df.columns]
    print(f"📋 Optional columns found: {present_optional}")
    
    # Check for missing values
    null_counts = df[required_cols + [c for c in optional_cols if c in df.columns]].isnull().sum()
    if null_counts.sum() > 0:
        print(f"\n⚠️ Missing values detected:")
        print(null_counts[null_counts > 0])
    else:
        print("✅ No missing values in key columns")
    
    # Class distribution
    label_col = None
    for col in ['Depression_Label', 'PHQ8_Binary', 'binary']:
        if col in df.columns:
            label_col = col
            break
    
    if label_col:
        print(f"\n⚖️ Class Distribution ({label_col}):")
        dist = df[label_col].value_counts()
        print(dist)
        print(f"   Ratio: {dist.min()/dist.max():.2%} minority/majority")
    
    # Source distribution (if available)
    if 'Source' in df.columns:
        print(f"\n🌍 Data Source Distribution:")
        source_dist = df['Source'].value_counts()
        for src, cnt in source_dist.items():
            # Mask actual names - display as generic
            generic_name = f"Source_{source_dist.index.tolist().index(src) + 1}"
            print(f"   {generic_name}: {cnt} samples")
else:
    print(f"❌ Labels file not found: {LABELS}")
    df = None

# Count H5 files
h5_files = glob.glob(f"{DATA_DIR}/*.h5")
print(f"\n📦 H5 Files Found: {len(h5_files)}")

if df is not None and len(h5_files) > 0:
    # Cross-reference
    h5_pids = set([Path(f).stem for f in h5_files])
    label_pids = set(df['Participant_ID'].astype(str))
    
    matched = h5_pids & label_pids
    h5_only = h5_pids - label_pids
    label_only = label_pids - h5_pids
    
    print(f"\n🔗 Cross-Reference Results:")
    print(f"   ✅ Matched (have both H5 & label): {len(matched)}")
    print(f"   ⚠️ H5 only (no label): {len(h5_only)}")
    print(f"   ⚠️ Label only (no H5): {len(label_only)}")

## Step 6: H5 File Structure Verification (108-Step Compliance)

In [ ]:
import h5py
import random

print("🔬 H5 Structure Verification...\n")

# Expected modalities and their feature groups
EXPECTED_MODALITIES = {
    'audio': ['mfcc', 'mel_spectrogram', 'prosody', 'wav2vec'],
    'video': ['facial_landmarks', 'action_units', 'gaze', 'pose'],
    'text': ['bert_embeddings', 'sentiment', 'linguistic'],
    'tabular': ['demographics', 'metadata']
}

def verify_h5_structure(h5_path):
    """Verify H5 file structure and count features."""
    issues = []
    feature_count = 0
    
    try:
        with h5py.File(h5_path, 'r') as f:
            # Count all datasets (features)
            def count_datasets(name, obj):
                nonlocal feature_count
                if isinstance(obj, h5py.Dataset):
                    feature_count += 1
            
            f.visititems(count_datasets)
            
            # Check for key modalities
            root_keys = list(f.keys())
            
    except Exception as e:
        issues.append(f"Cannot open: {e}")
        return None, issues
    
    return feature_count, issues

# Sample verification (check random subset)
sample_size = min(10, len(h5_files))
sample_files = random.sample(h5_files, sample_size) if h5_files else []

print(f"📝 Sampling {sample_size} H5 files for verification...\n")

results = []
for h5_path in sample_files:
    pid = Path(h5_path).stem
    feat_count, issues = verify_h5_structure(h5_path)
    
    if issues:
        status = "❌"
        print(f"{status} {pid}: {issues}")
    else:
        # Check 108-step compliance
        if feat_count >= 100:  # Allow some tolerance
            status = "✅"
        else:
            status = "⚠️"
        print(f"{status} {pid}: {feat_count} features")
    
    results.append({'pid': pid, 'features': feat_count, 'status': status})

# Summary
if results:
    valid_count = sum(1 for r in results if r['status'] == '✅')
    print(f"\n📊 Verification Summary:")
    print(f"   ✅ Compliant: {valid_count}/{len(results)}")
    print(f"   Average features: {np.mean([r['features'] for r in results if r['features']]):.0f}")

## Step 7: Label Consistency Validation

In [ ]:
print("🏷️ Label Consistency Validation...\n")

if df is not None:
    consistency_issues = []
    
    # Check PHQ8 Score range (should be 0-24)
    if 'PHQ8_Score' in df.columns:
        min_score = df['PHQ8_Score'].min()
        max_score = df['PHQ8_Score'].max()
        print(f"📈 PHQ-8 Score Range: [{min_score}, {max_score}]")
        
        if min_score < 0 or max_score > 24:
            consistency_issues.append(f"PHQ-8 scores out of range [0-24]: min={min_score}, max={max_score}")
        else:
            print("   ✅ Valid range [0-24]")
    
    # Check binary label consistency with PHQ8 score
    if 'PHQ8_Score' in df.columns and 'PHQ8_Binary' in df.columns:
        derived_binary = (df['PHQ8_Score'] >= 10).astype(int)
        mismatches = (derived_binary != df['PHQ8_Binary']).sum()
        
        if mismatches > 0:
            consistency_issues.append(f"{mismatches} mismatches between PHQ8_Score >= 10 and PHQ8_Binary")
            print(f"⚠️ {mismatches} label mismatches detected")
        else:
            print("✅ Binary labels consistent with PHQ-8 threshold (>=10)")
    
    # Check for duplicate Participant IDs
    dup_count = df['Participant_ID'].duplicated().sum()
    if dup_count > 0:
        consistency_issues.append(f"{dup_count} duplicate Participant_IDs")
        print(f"⚠️ {dup_count} duplicate IDs found")
    else:
        print("✅ No duplicate Participant IDs")
    
    # Summary
    print(f"\n📋 Consistency Check Summary:")
    if consistency_issues:
        print(f"   ⚠️ Issues found: {len(consistency_issues)}")
        for issue in consistency_issues:
            print(f"      - {issue}")
    else:
        print("   ✅ All consistency checks passed!")
else:
    print("❌ Cannot validate labels - DataFrame not loaded")

## Step 8: Load Model & Verify Checkpoints

In [ ]:
import torch
from pathlib import Path

print("🧠 Model & Checkpoint Verification...\n")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Using device: {device}")

# Import model classes
try:
    from config.model_config import H5Config, ComputeTier
    from src.models.h5_omnifusion import H5OmniFusion
    from src.data.h5_dataset import create_h5_dataloaders_kfold
    print("✅ Model modules imported successfully")
except Exception as e:
    print(f"❌ Import error: {e}")
    raise

# Find checkpoints
def find_checkpoints():
    """Search for Phase 10 checkpoints."""
    search_paths = [
        CHECKPOINT_DIR,
        f"{DATA_ROOT}/checkpoints_phase9",
        f"{DATA_ROOT}/checkpoints"
    ]
    
    for path in search_paths:
        if os.path.exists(path):
            ckpts = list(Path(path).glob("*_best.pt"))
            if ckpts:
                print(f"📁 Found checkpoints in: {path}")
                return ckpts, path
    
    # Recursive search as fallback
    print("🔍 Searching recursively...")
    ckpts = list(Path(DATA_ROOT).rglob("*_best.pt"))
    return ckpts, DATA_ROOT if ckpts else None

checkpoints, ckpt_dir = find_checkpoints()

if not checkpoints:
    print("❌ No checkpoints found!")
    best_ckpt = None
else:
    print(f"\n📦 Found {len(checkpoints)} checkpoint(s):")
    for ckpt in sorted(checkpoints):
        size_mb = ckpt.stat().st_size / (1024*1024)
        print(f"   {ckpt.name} ({size_mb:.1f} MB)")
    
    # Use latest checkpoint
    checkpoints.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    best_ckpt = checkpoints[0]
    print(f"\n👉 Using: {best_ckpt.name}")

# Initialize model
print(f"\n🔧 Initializing {TIER} tier model...")
config = H5Config.from_tier(ComputeTier(TIER))
model = H5OmniFusion(config)
param_count = sum(p.numel() for p in model.parameters())
print(f"   Parameters: {param_count/1e6:.2f}M")

# Load weights
if best_ckpt:
    try:
        checkpoint = torch.load(best_ckpt, map_location=device, weights_only=False)
        
        if isinstance(checkpoint, dict):
            if 'model_state_dict' in checkpoint:
                state_dict = checkpoint['model_state_dict']
            elif 'state_dict' in checkpoint:
                state_dict = checkpoint['state_dict']
            else:
                state_dict = checkpoint
        else:
            state_dict = checkpoint
            
        model.load_state_dict(state_dict, strict=False)
        print("✅ Model weights loaded successfully")
    except Exception as e:
        print(f"❌ Error loading weights: {e}")

model.to(device)
model.eval()
print("✅ Model ready for evaluation")

## Step 9: Comprehensive Evaluation

In [ ]:
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_absolute_error
)
from tqdm import tqdm

print("📊 Running Comprehensive Evaluation...\n")

# Helper function for moving data to device
def to_device(data, device):
    if isinstance(data, torch.Tensor):
        return data.to(device)
    elif isinstance(data, dict):
        return {k: to_device(v, device) for k, v in data.items()}
    elif isinstance(data, list):
        return [to_device(v, device) for v in data]
    return data

# Create test dataloader
if best_ckpt:
    try:
        train_loader, val_loader, test_loader = create_h5_dataloaders_kfold(
            h5_dir=DATA_DIR,
            labels_csv=LABELS,
            batch_size=BATCH_SIZE,
            fold_idx=0,
            n_folds=5,
            max_seq_len=MAX_SEQ_LEN
        )
        print(f"✅ Test set size: {len(test_loader.dataset)}")
    except Exception as e:
        print(f"❌ Error creating dataloaders: {e}")
        test_loader = None

# Run inference
all_preds = []
all_targets = []
all_probs = []
all_scores = []  # For PHQ-8 regression if available
all_pred_scores = []

if best_ckpt and test_loader:
    print("\n🔮 Running inference...")
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating"):
            try:
                # Prepare inputs
                inputs = {k: to_device(v, device) for k, v in batch.items() 
                          if k not in ['label', 'labels', 'target', 'targets']}
                
                # Get targets
                if 'label' in batch:
                    targets = to_device(batch['label']['binary'], device)
                    if 'phq_score' in batch['label']:
                        scores = batch['label']['phq_score'].cpu().numpy()
                        all_scores.extend(scores)
                elif 'labels' in batch:
                    targets = to_device(batch['labels']['binary'], device)
                elif 'targets' in batch:
                    targets = to_device(batch['targets']['binary'], device)
                else:
                    continue
                
                # Forward pass
                outputs = model(inputs)
                probs = outputs[0]['binary_prob']
                preds = (probs >= 0.5).long()
                
                # Collect results
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(targets.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())
                
                # Get predicted scores if available
                if 'phq_score' in outputs[0]:
                    all_pred_scores.extend(outputs[0]['phq_score'].cpu().numpy())
                    
            except Exception as e:
                print(f"⚠️ Batch error: {e}")
                continue
    
    print(f"\n✅ Inference complete: {len(all_preds)} predictions")
else:
    print("⚠️ Skipping evaluation - no model or dataloader")

## Step 10: Results Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve

if len(all_targets) > 0:
    y_true = np.array(all_targets)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)
    
    # Calculate metrics at default threshold (0.5)
    print("="*50)
    print("📊 RESULTS AT DEFAULT THRESHOLD (0.50)")
    print("="*50)
    
    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)
    
    print(f"  F1 Score:     {f1:.4f}")
    print(f"  Precision:    {precision:.4f}")
    print(f"  Recall:       {recall:.4f}")
    print(f"  Accuracy:     {accuracy:.4f}")
    
    # AUC-ROC (requires both classes)
    if len(set(y_true)) > 1:
        auc = roc_auc_score(y_true, y_prob)
        print(f"  AUC-ROC:      {auc:.4f}")
    else:
        auc = None
        print("  AUC-ROC:      N/A (single class in test set)")
    
    # PHQ-8 MAE if available
    if all_scores and all_pred_scores:
        mae = mean_absolute_error(all_scores, all_pred_scores)
        print(f"  PHQ-8 MAE:    {mae:.4f}")
    
    print("="*50)
    
    # Classification Report
    print("\n📋 Classification Report:")
    print(classification_report(y_true, y_pred, target_names=['Non-Depressed', 'Depressed']))
    
    # Visualizations
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['Non-Dep', 'Depressed'],
                yticklabels=['Non-Dep', 'Depressed'])
    axes[0].set_title(f'Confusion Matrix (F1={f1:.3f})')
    axes[0].set_ylabel('True Label')
    axes[0].set_xlabel('Predicted Label')
    
    # ROC Curve
    if auc:
        fpr, tpr, thresholds = roc_curve(y_true, y_prob)
        axes[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'Model (AUC={auc:.3f})')
        axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
        axes[1].fill_between(fpr, tpr, alpha=0.2)
        axes[1].set_xlabel('False Positive Rate')
        axes[1].set_ylabel('True Positive Rate')
        axes[1].set_title('ROC Curve')
        axes[1].legend(loc='lower right')
        axes[1].grid(True, alpha=0.3)
    else:
        axes[1].text(0.5, 0.5, 'ROC not available\n(single class)', 
                     ha='center', va='center', fontsize=14)
        axes[1].set_title('ROC Curve')
    
    plt.tight_layout()
    plt.savefig('/content/phase10_evaluation_plots.png', dpi=150)
    plt.show()
    print("\n📊 Plots saved to: /content/phase10_evaluation_plots.png")
else:
    print("❌ No predictions to visualize")

## Step 11: Threshold Optimization Analysis

In [ ]:
if len(all_probs) > 0:
    print("🎯 Threshold Optimization Analysis...\n")
    
    y_true = np.array(all_targets)
    y_prob = np.array(all_probs)
    
    # Test range of thresholds
    thresholds = np.arange(0.20, 0.80, 0.02)
    results = []
    
    for t in thresholds:
        y_pred_t = (y_prob >= t).astype(int)
        f1_t = f1_score(y_true, y_pred_t, zero_division=0)
        prec_t = precision_score(y_true, y_pred_t, zero_division=0)
        rec_t = recall_score(y_true, y_pred_t, zero_division=0)
        acc_t = accuracy_score(y_true, y_pred_t)
        results.append({'threshold': t, 'f1': f1_t, 'precision': prec_t, 'recall': rec_t, 'accuracy': acc_t})
    
    results_df = pd.DataFrame(results)
    
    # Find optimal thresholds for different objectives
    best_f1_idx = results_df['f1'].idxmax()
    best_f1_row = results_df.loc[best_f1_idx]
    
    print("="*50)
    print(f"✨ OPTIMAL THRESHOLD (Max F1)")
    print("="*50)
    print(f"  Threshold:    {best_f1_row['threshold']:.2f}")
    print(f"  F1 Score:     {best_f1_row['f1']:.4f}")
    print(f"  Precision:    {best_f1_row['precision']:.4f}")
    print(f"  Recall:       {best_f1_row['recall']:.4f}")
    print(f"  Accuracy:     {best_f1_row['accuracy']:.4f}")
    print("="*50)
    
    # Find balanced threshold (F1-Precision-Recall equilibrium)
    results_df['balance'] = results_df.apply(
        lambda r: min(r['precision'], r['recall']) / max(r['precision'], r['recall'] + 0.001), axis=1)
    balanced_idx = results_df['balance'].idxmax()
    balanced_row = results_df.loc[balanced_idx]
    
    print(f"\n⚖️ BALANCED THRESHOLD (Min P/R Gap):")
    print(f"  Threshold:    {balanced_row['threshold']:.2f}")
    print(f"  F1 Score:     {balanced_row['f1']:.4f}")
    print(f"  Precision:    {balanced_row['precision']:.4f}")
    print(f"  Recall:       {balanced_row['recall']:.4f}")
    
    # Plot threshold analysis
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(results_df['threshold'], results_df['f1'], 'b-', label='F1', linewidth=2)
    plt.plot(results_df['threshold'], results_df['precision'], 'g--', label='Precision')
    plt.plot(results_df['threshold'], results_df['recall'], 'r--', label='Recall')
    plt.axvline(x=best_f1_row['threshold'], color='blue', linestyle=':', alpha=0.7)
    plt.xlabel('Threshold')
    plt.ylabel('Score')
    plt.title('Metrics vs Threshold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.plot(results_df['threshold'], results_df['accuracy'], 'm-', linewidth=2)
    plt.axvline(x=best_f1_row['threshold'], color='blue', linestyle=':', alpha=0.7, label=f'Best F1 @ {best_f1_row["threshold"]:.2f}')
    plt.xlabel('Threshold')
    plt.ylabel('Accuracy')
    plt.title('Accuracy vs Threshold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('/content/phase10_threshold_analysis.png', dpi=150)
    plt.show()
    print("\n📊 Threshold plots saved to: /content/phase10_threshold_analysis.png")
    
    # Target check
    print("\n🎯 Target Check (Goals: F1≥0.85, Recall≥0.85, Precision≥0.85):")
    print(f"  {'✅' if best_f1_row['f1'] >= 0.85 else '❌'} F1:        {best_f1_row['f1']:.4f}")
    print(f"  {'✅' if best_f1_row['recall'] >= 0.85 else '❌'} Recall:    {best_f1_row['recall']:.4f}")
    print(f"  {'✅' if best_f1_row['precision'] >= 0.85 else '❌'} Precision: {best_f1_row['precision']:.4f}")
else:
    print("❌ No probability predictions for threshold analysis")

## Step 12: Generate Final Report & Save Results

In [ ]:
from datetime import datetime

print("📝 Generating Final Report...\n")

# Collect all results
report = {
    'timestamp': datetime.now().isoformat(),
    'phase': 10,
    'tier': TIER,
    'checkpoint': str(best_ckpt) if best_ckpt else None,
    'test_samples': len(all_preds) if all_preds else 0,
}

if len(all_preds) > 0:
    # Add optimal metrics
    report['optimal_threshold'] = float(best_f1_row['threshold'])
    report['metrics'] = {
        'f1': float(best_f1_row['f1']),
        'precision': float(best_f1_row['precision']),
        'recall': float(best_f1_row['recall']),
        'accuracy': float(best_f1_row['accuracy']),
    }
    if auc:
        report['metrics']['auc_roc'] = float(auc)

# Save predictions CSV
if all_preds:
    pred_df = pd.DataFrame({
        'true_label': all_targets,
        'predicted_label': all_preds,
        'probability': all_probs
    })
    pred_csv_path = '/content/phase10_predictions.csv'
    pred_df.to_csv(pred_csv_path, index=False)
    print(f"✅ Predictions saved: {pred_csv_path}")
    
    # Copy to Drive
    drive_path = f"{DATA_ROOT}/phase10_predictions.csv"
    !cp {pred_csv_path} "{drive_path}"
    print(f"✅ Copied to Drive: {drive_path}")

# Save report JSON
import json
report_path = '/content/phase10_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print(f"✅ Report saved: {report_path}")

# Copy plots to Drive
!cp /content/phase10_evaluation_plots.png "{DATA_ROOT}/phase10_evaluation_plots.png" 2>/dev/null || true
!cp /content/phase10_threshold_analysis.png "{DATA_ROOT}/phase10_threshold_analysis.png" 2>/dev/null || true

# Final Summary
print("\n" + "="*60)
print("🏆 PHASE 10 EVALUATION COMPLETE")
print("="*60)
print(f"📁 Checkpoint Used: {best_ckpt.name if best_ckpt else 'None'}")
print(f"📊 Test Samples: {len(all_preds)}")
if 'metrics' in report:
    print(f"\n🎯 Best Results (Threshold={report['optimal_threshold']:.2f}):")
    print(f"   F1 Score:     {report['metrics']['f1']:.4f}")
    print(f"   Precision:    {report['metrics']['precision']:.4f}")
    print(f"   Recall:       {report['metrics']['recall']:.4f}")
    print(f"   Accuracy:     {report['metrics']['accuracy']:.4f}")
    if 'auc_roc' in report['metrics']:
        print(f"   AUC-ROC:      {report['metrics']['auc_roc']:.4f}")
print("="*60)